In [ ]:
import numpy as np
from pathlib import Path
import pysacsnow as pss
import pandas as pd
import plotly.express as px

print(pss.__version__)

### Option 1: Manual Control Running all of the model and passing data back and forth between each model, each timestep

In [ ]:
basin_name = "16103000"

In [ ]:
# Set yaml file paths (make a little more dynamic based on basin_name)
sac_yaml_file = Path(f"../configurations/sac/sac.control.{basin_name}.yaml").resolve()
uhg_yaml_file = Path(f"../configurations/unithg/uhg.control.{basin_name}.yaml").resolve()

In [ ]:
# Initalize controllers
sac_control = pss.Controller.load_from_yaml(sac_yaml_file, verbose=False)
uhg_control = pss.Controller.load_from_yaml(uhg_yaml_file, verbose=False)

In [ ]:
# Initialize models
sac = pss.SACSMA(control=sac_control, calc_method="numpy")
uhg = pss.UHG(control=uhg_control, calc_method="numpy")

In [ ]:
sac.reset()
uhg.reset()

In [ ]:
# Run model for all time steps
### Option 1: Manual Control Running all of the model and passing data back and forth between each model, each timestep
n_times = sac_control.n_times
for tt in range(n_times):   
    
    # Advance controllers (i.e., time step to next time)
    sac_control.advance()
    uhg_control.advance()

    # calculate sac model
    sac.advance()
    sac.calculate()
    sac.output()

    # Pass sac outputs to uhg model inputs and calculate uhg model
    uhg.hru_tci[:] = sac.tci[:]
    uhg.calculate()
    uhg.output()


In [ ]:
# If you want, you can save individual model outputs as well
sac.finalize()
uhg.finalize()

In [ ]:
# Gather total flow from UHG model (sum across all hrus)
q_output = uhg.buffer_states["q"]

times = uhg.value_time_state[:-1]
total_flow = np.sum(q_output, axis=1)[:-1]
flow_df = pd.DataFrame({"flow_sim": total_flow}, index=pd.to_datetime(times))

In [ ]:
import holoviews as hv
hv.extension('bokeh') 
from bokeh.models import DatetimeTickFormatter, MonthsTicker

In [ ]:
# Create plot of daily flow vs qme in observations
obs_data = pd.read_csv(f"../data/observations/QIN_{basin_name}.csv", parse_dates=["time"])
obs_data.head()
obs_data["time"] = pd.to_datetime(obs_data["time"])
obs_data.set_index("time", inplace=True)
# Trim the observed data to the same time period as the simulated data
obs_df_trim = obs_data[flow_df.index[0]:flow_df.index[-1]]

In [ ]:
df_compare = pd.concat([flow_df, obs_df_trim], axis=1)
df_compare.columns = ["flow_sim", "flow_obs"]
df_compare = df_compare.reset_index().rename(columns={'index':'date'})
area_km2 = 47.94
df_compare['flow_obs'] = df_compare['flow_obs'] * area_km2 * 1000 / 3600
df_compare

In [ ]:
curve = hv.Curve(df_compare, kdims='date', vdims='flow_sim', label='sim').opts(
    width=900,
    height=400,
    xlabel='Datetime',
    ylabel='cms',
    tools=['hover'],
    xrotation=90,    
)
curve_obs = hv.Curve(df_compare, kdims='date', vdims='flow_obs', label='obs').opts(
    width=900,
    height=400,
    xlabel='Datetime',
    tools=['hover'],
    xrotation=90,    
)

curve * curve_obs

In [ ]:
# Stats
# Calculate the root mean squared error (RMSE)
rmse = np.sqrt(np.mean((df_compare["flow_sim"] - df_compare["flow_obs"])**2))
print(f"RMSE: {rmse:.2f}")
# Calculate percent bias (PBIAS)
pbias = 100 * np.sum(df_compare["flow_sim"] - df_compare["flow_obs"]) / np.sum(df_compare["flow_obs"])
print(f"PBIAS: {pbias:.2f}")
# Calculate Nash Sutcliffe Efficiency (NSE)
nse = 1 - np.sum((df_compare["flow_obs"] - df_compare["flow_sim"])**2) / np.sum((df_compare["flow_obs"] - np.mean(df_compare["flow_obs"]))**2)
print(f"NSE: {nse:.2f}")
# Calculate the coefficient of determination (R^2)
r_squared = np.corrcoef(df_compare["flow_sim"], df_compare["flow_obs"])[0, 1]**2
print(f"R^2: {r_squared:.2f}")

In [ ]:
del sac, uhg

### Option 2: Use custom helper workflow class that you make (like the one I am attaching) to make things a little more generalized

In [ ]:
from workflow import ModelWorkflow
# import importlib
# importlib.reload(ModelWorkflow)

In [ ]:

pss_wf = ModelWorkflow(
    basin_name = basin_name,
    snow17_yaml_file = None,
    sac_yaml_file = sac_yaml_file,
    uhg_yaml_file = uhg_yaml_file,
    lagk_yaml_file = None,
    consuse_yaml_file = None,
)

# Initialize models
# The initialize step is helpful to pre-load all model parameters and
# forcing data before running the models. And allows the user to modify models/parameters on the fly 
# (this is probably how you would do spotpy calibration for example).
pss_wf.initialize()

# Run the models
sim_flow = pss_wf.run(
    weights = None,
    cms_losses = None,
    return_mean_q = True
)

In [ ]:
sim_flow

In [ ]:
# Create plot of daily flow vs qme in observations
obs_data = pd.read_csv(f"../data/observations/QME_{basin_name}.csv", parse_dates=["Date"])
obs_data.head()
obs_data["Date"] = pd.to_datetime(obs_data["Date"])
obs_data.set_index("Date", inplace=True)
# Trim the observed data to the same time period as the simulated data
obs_df_trim = obs_data[sim_flow.index[0]:sim_flow.index[-1]]

df_compare = pd.concat([sim_flow, obs_df_trim], axis=1)
df_compare.columns = ["flow_sim", "flow_obs"]

# Plot the comparison of simulated and observed flow
fig = px.line(df_compare, x=df_compare.index, y=["flow_sim", "flow_obs"], title=f"Observed and Simulated Flow at {basin_name}")
# change size of the figure
fig.update_layout(width=1200, height=600)
fig.update_xaxes(rangeslider_visible=True)

In [ ]:
fig.show()

In [ ]:
df_compare[["flow_sim", "flow_obs"]].describe()

In [ ]:
df_compare = df_compare.reset_index().rename(columns={'index':'date'})

In [ ]:
import holoviews as hv

hv.extension("bokeh")

curve_sim = hv.Curve(
    df_compare.reset_index(),
    kdims=["date"],
    vdims=["flow_sim"],
    label="Simulated Flow",
)

curve_obs = hv.Curve(
    df_compare.reset_index(),
    kdims=["date"],
    vdims=["flow_obs"],
    label="Observed Flow",
)

plot = (curve_sim * curve_obs).opts(
    width=1200,
    height=600,
    title=f"Observed and Simulated Flow at {basin_name}",
    xlabel="date",
    ylabel="Flow",
    tools=["hover"],
    legend_position="top_left",
    show_grid=True,
)

plot